In [599]:
from pathlib import Path
import sys
import random

# test_dictionary.ipynb is inside backend/test/
backend_dir = Path.cwd().parent

# Make sure this is the directory containing llm.py
assert (backend_dir / "llm.py").exists(), backend_dir

sys.path.insert(0, str(backend_dir))

import pandas as pd
import json
from llm import detection
from main import lookup_span
nlp = spacy.load("it_core_news_lg")


In [600]:
df = pd.read_csv("set_n_gente.csv")


In [601]:
df.head()

,Unnamed: 0,SRC,REF-G,REF-N
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi..."


In [602]:
df["detection"] = None
df["rewriting_cv_0"] = None
df["rewriting_cv_1"] = None
df["rewriting_io"] = None

In [603]:
df_test = df[:5]
df_test.head()

,Unnamed: 0,SRC,REF-G,REF-N,detection,rewriting_cv_0,rewriting_cv_1,rewriting_io
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...,None,None,None,None
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,None,None,None,None
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...,None,None,None,None
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...,None,None,None,None
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi...",None,None,None,None


In [604]:
# -- reading of the .json lookup table for morphosyntactic rewrites -------
dictionary_path = "../dictionary.json"

with open(dictionary_path, "r", encoding="utf-8") as f:
    LOOKUP_TABLE: dict = json.load(f)

In [605]:
GENDERED_POS = {"noun", "adj", "verb", "det", "adp"}
#FLECTABLE_POS = {"noun", "adj", "verb", "aux", "det", "num", "pron", "adp"}
IGNORED_POS = {"punct", "space", "sym"}
STRATEGIES = ["CV-0", "CV-1", "IO"]
NEUTRAL = {"IO", "IV"}
FEMININE = {"CV"}
PUNTEGGIATURA = [".", ",", "!", "?"]

In [606]:
def find_forms(span, strategy):
    if len(strategy.split("-")) > 1:
        strategy, _ = strategy.split("-")
    if span is None or len(span) == 0:
        return None
    results = {}
    gender = "feminine" if strategy in FEMININE else "neutral"
    for token in span:
        word = token.text
        pos = token.pos_.lower()
        if pos == "adp":
            pos = "det"
        form = word
        if pos in GENDERED_POS:
            # If any gendered for is not found, we resort to the llm call
            entry = LOOKUP_TABLE.get(word)
            if entry is None:
                return None

            try:
                form = entry[pos][gender][0]["form"]
            except (KeyError, IndexError, TypeError):
                return None

        results[word] = {
                    "pos": pos,
                    "form": form,
                    "changed": form != word,
                    }    

    return results

In [607]:
def render_tokens(tokens, forms, feminine=False):
    """Render tokens while preserving their original spacing."""
    parts = []

    for token in tokens:
        if feminine:
            form = forms[token.text]["form"]
        else:
            form = token.text

        parts.append(form + token.whitespace_)

    return "".join(parts).rstrip()

In [608]:
def combine_forms(masc, fem, preference, det=True):
    conjunction = "e" if det else "o"

    masc_text = "".join(t.text + t.whitespace_ for t in masc).rstrip()
    fem_text = "".join(fem).rstrip()
    trailing_space = masc[-1].whitespace_ if masc else ""
    if preference == "feminine":
        sent = f"{fem_text} {conjunction} {masc_text}"
    else:
        sent = f"{masc_text} {conjunction} {fem_text}"
    sent += trailing_space
    return list(nlp(sent))

In [609]:
def to_other_form(span, forms):
    other_form = ""
    for token in span:
        w = token.text
        if forms[w]["pos"] != "punct":
            other_form += " "
        other_form += forms[w]["form"]
    return other_form

In [610]:
def handle_cv_long(masc_doc, forms, gender_precedence):
    output = []
    curr_masc = []
    curr_fem = []
    for token_m in masc_doc:
        word = token_m.text
        feminine = forms[word]["form"]
        # Preserve the original token's whitespace in the feminine version.
        token_f = feminine + (token_m.whitespace_ if not feminine.endswith("'") else "")
        if feminine != word:
            curr_masc.append(token_m)
            curr_fem.append(token_f)
        else:
            if curr_masc and curr_fem:
                det = False if (curr_masc and curr_masc[0].text == "un") else True
                output.extend(combine_forms(curr_masc, curr_fem, gender_precedence, det))
                curr_masc = []
                curr_fem = []
            output.append(token_m)
    # Flush any remaining changed tokens after the loop.
    if curr_masc and curr_fem:
        det = False if curr_masc and curr_masc[0] == "un" else True
        output.extend(
            combine_forms(
                curr_masc,
                curr_fem,
                gender_precedence,
                det
            )
        )
    return "".join(t.text + t.whitespace_ for t in output).strip()

In [611]:
def handle_cv_bars(masc_doc, forms, gender_precedence):
    output = []
    for token_m in masc_doc:
        word = token_m.text
        fem = forms[word]["form"]
        if fem != word:
            token = f"{fem}/{word}" if gender_precedence == "feminine" else f"{word}/{fem}"
            token += token_m.whitespace_ if not fem.endswith("'") else ""
            output.append(token)
        else:
            token = word + token_m.whitespace_
            output.append(token)
    
    return "".join(t for t in output).strip() 


In [612]:
def handle_io(masc_doc, forms):
    output = []
    for token_m in masc_doc:
        word = token_m.text
        neut = forms[word]["form"]
        if neut != word:
            token = neut + token_m.whitespace_
            output.append(token)
        else:
            token = word + token_m.whitespace_
            output.append(token)
    
    return "".join(t for t in output).strip() 

In [613]:
def rewrite(masc_doc, span_txt, forms, strategy):
    gender_precedence = random.sample(["masculine", "feminine"], 1)[0]
    if strategy == "CV-0":
        return handle_cv_long(masc_doc, forms, gender_precedence)
    if strategy == "CV-1":
        return handle_cv_bars(masc_doc, forms, gender_precedence)
    if strategy == "IO":
        return handle_io(masc_doc, forms)
    

In [614]:
def transform_sentence(spans, sentence, strategy):
    doc = nlp(sentence)
    output = []
    curr_idx = 0

    for span in sorted(spans, key=lambda s: s.start_char):
        start = span.start_char
        end = span.end_char
        # Avoid processing overlapping spans twice.
        if start < curr_idx:
            continue
        # Preserve text outside the detected span.
        output.append(sentence[curr_idx:start])
        nlp_span = doc.char_span(start, end, alignment_mode="expand")
        forms = find_forms(nlp_span, strategy)
        if forms is None:
            return "Almeno una parola di questa frase non è stata trovata nel dizionario!"
        else:
            output.append(rewrite(nlp_span, span.original_text, forms, strategy))
        curr_idx = end
    
    # Preserve the remainder after the last span.
    output.append(sentence[curr_idx:])
    return "".join(output)

In [615]:
for i, row in df.iterrows():
    sentence = row["REF-G"]
    # Perform detection
    spans = detection(sentence)
    rewritings = []
    output_detection = None
    if not spans:
        rewritings = [None, None, None]
    else:
        output_detection = " | ".join( [span.original_text for span in spans])
        for strategy in STRATEGIES:
            rewriting = transform_sentence(spans, sentence, strategy)
            rewritings.append(rewriting)

    df.loc[i, "detection"] = output_detection
    df.loc[i, "rewriting_cv_0"] = rewritings[0]
    df.loc[i, "rewriting_cv_1"] = rewritings[1]
    df.loc[i, "rewriting_io"] = rewritings[2]
    

2026-10-28 10:06:27,216 - llm - WARNING - Span does not start with B-UNFAIR: residenti
2026-10-28 10:06:27,382 - llm - WARNING - Span does not start with B-UNFAIR: ,
2026-10-28 10:06:27,460 - llm - WARNING - Span does not start with B-UNFAIR: or
2026-10-28 10:06:27,786 - llm - WARNING - Span does not start with B-UNFAIR: ono
2026-10-28 10:06:28,232 - llm - WARNING - Span does not start with B-UNFAIR: ,
2026-10-28 10:06:29,243 - llm - WARNING - Span does not start with B-UNFAIR: cittadini
2026-10-28 10:06:29,658 - llm - WARNING - Span does not start with B-UNFAIR: nie
2026-10-28 10:06:30,047 - llm - WARNING - Span does not start with B-UNFAIR: li
2026-10-28 10:06:30,535 - llm - WARNING - Span does not start with B-UNFAIR: ed
2026-10-28 10:06:31,061 - llm - WARNING - Span does not start with B-UNFAIR: ,
2026-10-28 10:06:31,061 - llm - WARNING - Span does not start with B-UNFAIR: seguono
2026-10-28 10:06:31,103 - llm - WARNING - Span does not start with B-UNFAIR: soli
2026-10-28 10:06:31,

In [616]:
df.head(10)

,Unnamed: 0,SRC,REF-G,REF-N,detection,rewriting_cv_0,rewriting_cv_1,rewriting_io
0,750,They have shown that European issues can be wo...,Essi hanno dimostrato come sia possibile lavor...,Hanno dimostrato come sia possibile lavorare a...,Essi hanno dimostrato | dei cittadini.,Essi hanno dimostrato e dimostrata come sia po...,Essi hanno dimostrato/dimostrata come sia poss...,Essi hanno dimostrat<ENDS> come sia possibile ...
1,751,As the European institutions acquire ever more...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,"sui cittadini,",Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...,Man mano che le Istituzioni europee ottengono ...
2,752,The proposed Charter is unique in the way it g...,Nella Carta che viene proposta i cittadini e i...,Nella Carta che viene proposta la cittadinanza...,i cittadini,Nella Carta che viene proposta le cittadine e ...,Nella Carta che viene proposta le/i cittadine/...,Nella Carta che viene proposta l<ENDP> cittadi...
3,753,Discrimination between European and third coun...,La discriminazione fra europei e residenti dei...,La discriminazione fra cittadinanza europea e ...,europei | residenti,La discriminazione fra europee e europei e res...,La discriminazione fra europee/europei e resid...,La discriminazione fra europe<ENDP> e resident...
4,754,"In addition, there are political gaps in the C...","Inoltre, la Carta presenta delle lacune politi...","Inoltre, la Carta presenta delle lacune politi...",None,None,None,None
5,755,"I am sad, indeed, to see the ranks of British ...",Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che il partito Co...,dei cittadini,Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che i Conservator...,Pertanto mi rammarica vedere che i Conservator...
6,756,Let us not merely adopt these rights but make ...,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...,dei nostri cittadini:,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...,Non limitiamoci semplicemente ad approvare tal...
7,757,The citizens of Europe have continued to be in...,I cittadini europei sono rimasti insufficiente...,La cittadinanza europea è rimasta insufficient...,I cittadini europei sono rimasti insufficiente...,Almeno una parola di questa frase non è stata ...,Almeno una parola di questa frase non è stata ...,Almeno una parola di questa frase non è stata ...
8,758,Just one last word to the President-in-Office:...,Ancora un accenno al signor Presidente del Con...,Ancora un accenno alla Presidenza del Consigli...,None,None,None,None
9,759,"It is true, as one of the speakers said, that ...","È vero, come ha detto qualche oratore, che ser...","È vero, come ha detto qualche persona che ha p...","oratore,","È vero, come ha detto qualche oratore e oratri...","È vero, come ha detto qualche oratrice/oratore...","È vero, come ha detto qualche orator<ENDS>, ch..."


In [618]:
df.to_csv("set_n_gente_rewritings.csv", encoding="utf-8")

In [538]:
df_test.head()

,Unnamed: 0,SRC,REF-G,REF-N,detection,rewriting_cv_0,rewriting_cv_1,rewriting_io
0,1137,"It goes without saying that citizens must, in ...","E, comunque sia, è ovvio che i cittadini debba...","E, comunque sia, è ovvio che la cittadinanza d...",i cittadini debbano esserne pienamente informati.,"E, comunque sia, è ovvio che i cittadini e le ...","E, comunque sia, è ovvio che le/i cittadine/ci...","E, comunque sia, è ovvio che l<ENDP> cittadin<..."
1,1373,The current desire to extend qualified majorit...,L'attuale volontà di estendere i voti a maggio...,L'attuale volontà di estendere i voti a maggio...,ai cittadini.,L'attuale volontà di estendere i voti a maggio...,L'attuale volontà di estendere i voti a maggio...,L'attuale volontà di estendere i voti a maggio...
2,1270,"I would like to reaffirm what my colleague, Ma...",Condivido appieno quanto ha appena detto il mi...,Condivido appieno quanto ha appena detto l'ono...,None,None,None,None
3,1083,We have been able to find a majority of MEPs t...,Siamo riusciti a trovare una maggioranza per r...,Ha funzionato il nostro sforzo per trovare una...,None,None,None,None
4,1036,This could involve in certain cases putting an...,"Ciò può comportare, in alcuni casi, la necessi...","Ciò può comportare, in alcuni casi, la necessi...",un interprete.,"Ciò può comportare, in alcuni casi, la necessi...","Ciò può comportare, in alcuni casi, la necessi...","Ciò può comportare, in alcuni casi, la necessi..."
